# Alarm 9200 — large mount offsets: episode map

Alarm id **9200** (`WARNING` / `OCS`, *"Large mount offset … Check TCS zero point"*) flags a
telescope mount-offset problem. It recurs in **episodes** — bursts of ~10–20 over a month or two,
then quiet for a year or more, then back again — cause unknown. This notebook pulls every 9200
occurrence, finds the **exposure** each one is about, looks up its **mount pointing**, and maps where
on the sky the episodes cluster.

Example message:
```
9200  WARNING  OCS  DESI (156743): Large mount offset: 163.007000 arcsec. Check TCS zero point (ask OA)
```

Everything goes through the `telemetry_mining` module — `find_alarms` (global alarm search),
`Exposure.db_row` (the exposure's DB record), and `Exposure.at_time` (the exposure open at a
timestamp) — so there is **no hand-written SQL** and **no access to the on-disk FITS files**: it's all
in the database, which means purged old-night exposures resolve fine.

**Which exposure?** A "large mount offset" is an *acquisition/slew* alarm — it fires **before** the
target exposure's shutter opens — so the exposure it's about is the one **named in the message**
(`DESI (<expid>)`), and we use that. `Exposure.at_time` is kept only as a **cross-check**: it returns
the exposure that was *running* at the alarm's timestamp, which for this alarm is typically the
*previous* one, precisely because the alarm precedes its target. The **offset** is only in the
message, so it's parsed from there.

**Run at KPNO** — needs the live database (`DOS_DB_*`).

In [ ]:
import os, sys, re

# Portable bootstrap: point DOS_TELEMETRY_MINING_DIR at your checkout's src/, else the default.
TM_DIR = os.getenv("DOS_TELEMETRY_MINING_DIR", os.path.expanduser("~/telemetry_mining-trunk/src"))
sys.path.insert(0, TM_DIR)

import pandas as pd
import matplotlib.pyplot as plt

from telemetry_mining import find_alarms, Exposure
from telemetry_mining.config import Config

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

cfg = Config.default()
print(f"site={cfg.site}, db={cfg.db_name or '?'}@{cfg.db_host or '?'}")

## 1. Every alarm 9200 occurrence

`find_alarms` is the global search over the alarm log (the complement of `Exposure.alarms()`, which
is per-exposure). `alarm_id` is the alarm *type* — distinct from the row primary key `id`. The
per-year counts are the "episodes".

In [ ]:
alarms = find_alarms(
    cfg,
    alarm_id=9200,
    columns=["id", "alarm_id", "time_recorded", "level", "component", "instance", "message"],
)
alarms["year"] = pd.to_datetime(alarms["time_recorded"], utc=True).dt.year

print(f"{len(alarms)} alarm-9200 occurrences", end="")
if len(alarms):
    print(f", {int(alarms.year.min())}–{int(alarms.year.max())}")
    print("\nby year (the episodes):")
    print(alarms.year.value_counts().sort_index().to_string())
else:
    print(" — none found (try find_alarms(cfg, component='OCS', message_like='%Large mount offset%'))")
alarms.head()

## 2. Exposure id and offset from the message

The offset only lives in the message; the exposure id (the acquisition target) is the `DESI (<expid>)`
number.

In [ ]:
OFFSET_RE = re.compile(r"offset:\s*([-\d.]+)\s*arcsec")
EXPID_RE  = re.compile(r"DESI\s*\((\d+)\)")

alarms["offset_arcsec"] = alarms["message"].str.extract(OFFSET_RE)[0].astype(float)
alarms["expid"]         = alarms["message"].str.extract(EXPID_RE)[0].astype("Int64")
print(f"parsed an expid for {int(alarms.expid.notna().sum())}/{len(alarms)} messages, "
      f"an offset for {int(alarms.offset_arcsec.notna().sum())}")
alarms[["time_recorded", "expid", "offset_arcsec", "message"]].head()

## 3. Time-match cross-check (one example)

`Exposure.at_time(when)` returns the exposure that was *running* at `when` — the inverse of
`Exposure.time_window`. For this acquisition alarm it's usually the exposure *before* the one named in
the message (the alarm precedes its target). We show it on a **single** alarm rather than looping:
each call scans `exposure.exposure` by `date_obs` (not the primary key), so it's a per-timestamp
lookup — fine once, but **do not run it in a tight loop over hundreds of alarms** (that's a
full-table scan each time). It's DB-only and not used for the pointing below.

In [ ]:
# ONE example -- Exposure.at_time scans by date_obs, so it's a single-timestamp lookup, not a loop.
ex = alarms.dropna(subset=["expid", "time_recorded"]).iloc[0]
running = Exposure.at_time(ex["time_recorded"], config=cfg)
print(f"alarm at {ex['time_recorded']}:  message id = {ex['expid']}  |  "
      f"running exposure then = {running.expid if running is not None else None}")
print("(the message id is the acquisition target; at_time returns the previous, running exposure — "
      "they differ because the alarm fires before its target opens)")

## 4. Mount pointing for each exposure (from the DB record)

`Exposure(expid).db_row` is the full `exposure.exposure` record — a **DB-only** lookup, so it works
even for old exposures whose FITS files have been purged (nothing here reads `/exposures/desi`). We
want `mountha`, `mountdec`, `mountra`, and (if present) `slewangl`. **These may be flat columns or
inside the `tcs` jsonb blob** — the FITS 8-character convention means it's probably `slewangl` (not
`slewangle`). The discovery cell prints what's actually there so you can confirm the names; the
extractor tries flat columns first, then the `tcs` blob.

In [ ]:
# one db_row per unique exposure id (DB-only; robust to missing/aborted exposures)
_rowcache = {}
def db_row_for(expid):
    if pd.isna(expid):
        return None
    expid = int(expid)
    if expid not in _rowcache:
        try:
            _rowcache[expid] = Exposure(expid, config=cfg).db_row
        except Exception:
            _rowcache[expid] = None
    return _rowcache[expid]

# discovery: where do the mount/slew fields live? (confirm/adjust names below)
sample = next((r for r in (db_row_for(x) for x in alarms["expid"].dropna().unique()) if r is not None), None)
if sample is not None:
    flat = [k for k in sample if any(t in k.lower() for t in ("mount", "slew"))]
    tcs0 = sample.get("tcs") if isinstance(sample.get("tcs"), dict) else {}
    print("flat columns with mount/slew :", flat)
    print("tcs jsonb keys with mount/slew:", [k for k in tcs0 if any(t in k.lower() for t in ("mount", "slew"))])
else:
    print("no exposure records resolved — check the expid parsing / DB")

In [ ]:
def pick(row, *names):
    """First non-null value among `names`, looking in the flat row then its tcs blob."""
    if row is None:
        return None
    tcs = row.get("tcs") if isinstance(row.get("tcs"), dict) else {}
    for n in names:
        if row.get(n) is not None:
            return row[n]
        if tcs.get(n) is not None:
            return tcs[n]
    return None

recs = []
for _, a in alarms.iterrows():
    row = db_row_for(a["expid"])
    recs.append(dict(
        expid=a["expid"], time_recorded=a["time_recorded"], year=a["year"],
        offset_arcsec=a["offset_arcsec"],
        mountha =pick(row, "mountha",  "mount_ha"),
        mountdec=pick(row, "mountdec", "mount_dec"),
        mountra =pick(row, "mountra",  "mount_ra"),
        slewangl=pick(row, "slewangl", "slewangle", "slew_angle"),
    ))
df = pd.DataFrame(recs)
print(f"{df.mountha.notna().sum()}/{len(df)} exposures resolved mount pointing "
      f"({int(df.mountha.isna().sum())} missing / purged / aborted)")

# optional: save the assembled table alongside the other data products
OUT = os.path.join(os.path.dirname(TM_DIR), "data", "alarm_9200_mount_offsets.csv")
df.to_csv(OUT, index=False)
print("wrote", OUT)
df.head()

## 5. Where do the episodes land?

Scatter of **mount HA vs mount Dec**, one point per alarm, colour-coded by **episode (year)**.

In [ ]:
plot = df.dropna(subset=["mountha", "mountdec"])
years = sorted(int(y) for y in plot["year"].dropna().unique())
cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(8, 6))
for i, y in enumerate(years):
    s = plot[plot["year"] == y]
    ax.scatter(s["mountha"], s["mountdec"], s=60, color=cmap(i % 10),
               edgecolor="k", linewidth=0.4, alpha=0.85, label=f"{y}  (n={len(s)})")
ax.set_xlabel("mount HA (deg)")
ax.set_ylabel("mount Dec (deg)")
ax.set_title("Alarm 9200 (large mount offset): mount pointing by episode")
ax.grid(alpha=0.3)
ax.legend(title="episode", fontsize=9)
plt.tight_layout()
plt.show()

And the **slew-angle distribution** across all episodes.

In [ ]:
sl = df["slewangl"].dropna()
fig, ax = plt.subplots(figsize=(7, 4.5))
if len(sl):
    ax.hist(sl, bins=30, color="#4f7fb8", edgecolor="k", linewidth=0.4)
ax.set_xlabel("slewangl (deg)")
ax.set_ylabel("count")
ax.set_title(f"Alarm 9200: slew-angle distribution (all episodes, n={len(sl)})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Notes.**

- **Message id vs running exposure** (§3) — they differ by ~1 because this is an *acquisition* alarm
  (it fires before the target opens); the message id is the target, and is what §4/§5 use.
- **Field names** (§4) — the discovery cell prints where `mount*`/`slew*` live; if the extractor comes
  back empty, add the real names to the `pick(...)` calls.
- **No FITS files touched** — everything is from the DB (`find_alarms`, `Exposure.db_row`,
  `Exposure.at_time`), never `/exposures/desi`, so purged old-night files don't matter.